In [1]:
import numpy as np

## Conversions

In [2]:
AU_to_km = 1.495978707e8 # AU to Km
day_to_sec = 86400.0 # Day to Seconds
arcsec_to_rad = np.deg2rad(1/3600) # Arcsecond to Radian

In [3]:
angle_diff = lambda a, b: (a - b + np.pi) % (2*np.pi) - np.pi #Calculates angle difference
stn_to_ecef = lambda stn, properties: np.array([R_earth * properties['cos'] * np.cos(np.deg2rad(properties['Long.'])), R_earth * properties['cos'] * np.sin(np.deg2rad(properties['Long.'])), R_earth * properties['sin']]) #Long to Earth Centered Earth Fixed

## Constants

In [4]:
sun_radius = 695700 # Radius of the Sun (km)
sun_mu = 1.32712440018e11  # Gravitational parameter of the Sun (km^3/s^2)

R_earth = 6378.137 # Radius of the Earth(km)
R_earth_polar = 6356.7523 # Polar radius of the Earth(km)

c_km_s = 299792.458 #Speed of light (s)

## Code

In [5]:
import nest_asyncio
nest_asyncio.apply()

![](https://cdn.hackclub.com/019d41e8-3c3d-7b1b-b982-1c8db02d2431/Screenshot%202026-03-22%20173922.png)

``` mermaid

    flowchart LR

        %% 1. Style Definitions
        classDef var fill:#1a3e1a,color:#a8d8a8,stroke:#4a8a4a,stroke-width:2px
        classDef quantity fill:#2d3748,color:#e2e8f0,stroke:#4a5568,stroke-width:1px
        classDef ghost fill:none,stroke:none,color:none

        %% 2. Subgraph Definitions
        subgraph N0["Minor Planet Center Requests"]
            N0a[get-orb]
        end

        subgraph N1["Minor Planet Center Responses"]
            N1a[orb]:::var
            D1a["Eros's Orbital Data"]:::ghost
        end

        subgraph N2[" "]
            N2a[x]:::var
            N2b[coefficient values like]:::ghost
            N2c[x, y, z, vx, vy, vz]:::quantity
        end

        %% 3. Node Definitions
        N3a[t0_mjd]:::var

        %% 4. Box Hiding 
        N2:::ghost
    
        N0a --mpc_orb--> N1a

        N1a --coefficient_values--> N2a
        N1a --epoch--> N3a

        N2a --> N2b --> N2c
        N2c --Convert Au/day to km/s and Au to km--> N2a
```

In [6]:
import requests

response = requests.get("https://data.minorplanetcenter.net/api/get-orb", json={"desig": "Eros"})

if response.ok:
    orb = response.json()[0]['mpc_orb']
else:
    print("Error: ", response.status_code, response.content)

In [7]:
orbital_elements = orb[0]['CAR']['coefficient_values']
#coefficient values x, y, z, vx, vy, vz
x = np.array(orbital_elements) 

In [8]:
x[:3] *= AU_to_km #convert to km
x[3:] *= AU_to_km / day_to_sec #convert to km/s

In [9]:
print(x)

[ 1.20144699e+08  1.48565328e+08  3.49924625e+07 -2.41244768e+01
  1.30481711e+01 -2.40794861e+00]


In [10]:
t0_mjd = orb[0]['epoch_data']['epoch']

![](https://cdn.hackclub.com/019d41e8-917b-7bcf-b4d8-77ade07498a3/Screenshot%202026-03-22%20184824.png)

$
a_{small} = \frac{G_{large}}{r^2} = \frac{\mu}{r^2}
$

### Newton's Universal Law of Gravitation in 3D
$
\vec{a_{small}} = \frac{Gm_{large}}{r^2} = \frac{\mu}{r^2}
$

In [11]:
kernels = {
    "lsk/": ['naif0012.tls',],
    
    "spk/": ['planets/de440.bsp', 
            'asteroids/codes_300ast_20100725.tf',],
    
    "pck/": ['earth_latest_high_prec.bpc', 
            'gm_de431.tpc',
            'pck00010.tpc',]
}

In [12]:
import os
import urllib.request

url = 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/'
kernels_loaded = os.listdir("../data/kernels")

for kernel_type, items in kernels.items():
    for item in items:
        item_name = item.split('/')[-1]
        if item_name in kernels_loaded:
            print(f"{item_name} aldready downloaded")
            continue
        else:
            print
            download_url = url + kernel_type + item
            print(f"Downloading {item_name} from {download_url}")
            urllib.request.urlretrieve(download_url, f"kernels/{item_name}")
            
print("All files done!")
        

naif0012.tls aldready downloaded
de440.bsp aldready downloaded
codes_300ast_20100725.tf aldready downloaded
earth_latest_high_prec.bpc aldready downloaded
gm_de431.tpc aldready downloaded
pck00010.tpc aldready downloaded
All files done!


In [13]:

import spiceypy as sp

# The meta kernel file contains entries pointing to the following SPICE kernels, which the user needs to download.
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440.bsp
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/earth_latest_high_prec.bpc
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de431.tpc
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/asteroids/codes_300ast_20100725.tf
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc

#   The following is the contents of a metakernel that was saved with
#   the name 'planetaryMetaK.txt'.
#   \begindata
#   KERNELS_TO_LOAD=(
#     'kernels/naif0012.tls',
#     'kernels/de440.bsp',
#     'kernels/earth_latest_high_prec.bpc',
#     'kernels/gm_de431.tpc',
#     'kernels/codes_300ast_20100725.tf',
#     'kernels/pck00010.tpc',
#   )
#   \begintext

sp.kclear()
sp.furnsh("../data/planetaryMetaK.txt")

In [14]:
print(sp.bodvrd('2000087', 'GM', 1)[1][0])

0.9899999999999999


$
F_g = -Gm_i\sum_{j=1, j \neq i}^n\frac{m_j}{r_{ji}^3}(r_{ji})
$

$
a_g = -\mu\sum_{}\frac{r_{ji}}{|r_{ji}|^3}
$

In [15]:
def n_body_ode(t, state):
    
    r = state[:3]
    v = state[3:]
    
    r_norm = np.linalg.norm(r)
    a = -sun_mu * r / r_norm**3
    
    ssbs = [
        # (name, mu)
        
        # Terrestrial
        ('1', 22031.78000000002), # MERCURY
        ('2',   324858.592), # VENUS
        ('3',   398600.435), # EARTH
        ('4',    42828.375), # MARS
        
        # Satellites
        ('301',               4902.800), # MOON
        
        # Asteroids
        ('2000001',              63.129999999999995),
        ('2000002',              13.73),
        ('2000004',              17.28999999999999),
        
        ('2000010',            5.78),               # Hygiea
        ('2000015',            2.10),               # Eunomia
        ('2000016',            1.81),               # Psyche
        ('2000029',            0.86),               # Amphitrite
        ('2000052',            1.5899999999999999), # Europa
        ('2000065',            0.9099999999999999), # Cybele
        ('2000087',            0.9899999999999999), # Sylvia
        ('2000088',            1.02),               # Thisbe
        ('2000511',            2.259999999999999),  # Davida
        ('2000704',            2.189999999999999),  # Interamnia
        
        # Jovian
        ('5', 126712764.1), # JUPITER
        ('6',  37940585.2), # SATURN
        ('7',  5794548.600000008), # URANUS
        ('8', 6836527.100580023), # NEPTUNE
    ]
    
    for name, mu in ssbs:
        
        #j is eros, i is planet
        
        planet, _ = sp.spkezr(name, t, 'J2000', 'NONE', 'SUN')
        
        r_i = planet[:3]
        r_i_norm = np.linalg.norm(r_i)
        
        r_ji = r - r_i
        r_ji_norm = np.linalg.norm(r_ji)
        
        # subtracting pull due to sun
        a += -mu * (r_ji / r_ji_norm**3 + r_i / r_i_norm**3)
    
    return np.concatenate((v, a)) # vx, vy, vz, ax, ay, az

In [16]:
from scipy.integrate import solve_ivp

In [17]:
response = requests.get("https://data.minorplanetcenter.net/api/get-obs", json={"desigs": ["Eros"], "output_format": ["ADES_DF"]})

if response.ok:
    obs_data = response.json()[0]['ADES_DF']
else:
    print("Error: ", response.status_code, response.content)


In [18]:
for n, obs in enumerate(obs_data):
    obs['obstime'] = sp.str2et(obs['obstime'])
    obs_data[n] = obs

In [19]:
t_epoch = sp.unitim(t0_mjd + 2400000.5, 'JDTDT', 'ET')  # MJD 61000 ≈ this date

for n, obs in enumerate(obs_data):
    if obs['obstime'] >= t_epoch:
        obs_index = n
        break
    
t0 = obs_data[obs_index]['obstime']
ti = obs_data[-1]['obstime']
    
print(f"Propagating from {obs_index}th observation at {t0} to last index at {ti}")

Propagating from 16810th observation at 817011380.798867 to last index at 833104441.1850272


In [20]:
ecl_to_j2000 = sp.pxform('ECLIPJ2000', 'J2000', t_epoch)
x = np.concatenate([ecl_to_j2000 @ x[:3], ecl_to_j2000 @ x[3:]])
x = solve_ivp(n_body_ode,
                (t_epoch, t0),
                x,
                method = "DOP853",
                rtol=1e-12,
                atol=1e-12,
                dense_output=True,).y[:, -1]

In [21]:
trajectory_solution = solve_ivp(n_body_ode,
                                (t0, ti),
                                x,
                                method = "DOP853",
                                rtol=1e-12,
                                atol=1e-12,
                                dense_output=True,).sol

In [22]:
# stations.csv downloaded from https://www.minorplanetcenter.net/iau/lists/ObsCodes.html

import csv

def get_stn_properties(stn):
    
    with open('../data/stations.csv', 'r', newline='\n') as stns:
        reader = csv.DictReader(stns)
        for row in reader:
            if row['Code'] == stn:
                return {
                    'Long.': float(row['Long.']) if row['Long.'].strip() != '' else None,
                    'cos': float(row['cos']) if row['cos'].strip() != '' else None,
                    'sin': float(row['sin']) if row['sin'].strip() != '' else None,
                }
    return None

In [23]:
def stn_to_ecef(stn, properties):
    
    lon = np.deg2rad(properties['Long.'])
    x = R_earth * properties['cos'] * np.cos(lon)
    y = R_earth * properties['cos'] * np.sin(lon)
    z = R_earth * properties['sin']
    
    return np.array([x, y, z])

In [24]:
def get_observer_pos_j2000(stn, t_obs, properties):
    r_ecef = stn_to_ecef(stn, properties)
        
    earth, _ = sp.spkezr('EARTH', t_obs, 'J2000', 'NONE', 'SUN')
    rot = sp.pxform('ITRF93', 'J2000', t_obs)
    r_obs_j2000 = rot @ r_ecef
    
    return earth[:3] + r_obs_j2000

In [25]:
VFCCLookupDefault = {
# (stn, astcat) : (ra, dec) arcsec 

# USNOA2.0 
('704', 'USNOA2')  : (0.63, 0.60),
('699', 'USNOA2')  : (0.62, 0.53),
('691', 'USNOA2')  : (0.30, 0.30),
('608', 'USNOA2')  : (0.61, 0.75),
('703', 'USNOA2')  : (0.69, 0.63),
('644', 'USNOA2')  : (0.29, 0.30),
('291', 'USNOA2')  : (0.46, 0.32),
('599', 'USNOA2')  : (0.39, 0.34),
('333', 'USNOA2')  : (0.55, 0.53),
('D35', 'USNOA2')  : (0.39, 0.38),

# USNOA1.0 
('704', 'USNOA1')  : (0.76, 0.73),
('691', 'USNOA1')  : (0.49, 0.46),

# USNOB1.0 
('699', 'USNOB1')  : (0.61, 0.54),
('644', 'USNOB1')  : (0.24, 0.20),
('691', 'USNOB1')  : (0.30, 0.28),
('291', 'USNOB1')  : (0.39, 0.26),

# UCAC1
('703', 'UCAC1')   : (0.63, 0.59),
('G96', 'UCAC1')   : (0.32, 0.27),
('E12', 'UCAC1')   : (0.50, 0.45),
('683', 'UCAC1')   : (0.79, 0.90),
('J75', 'UCAC1')   : (0.41, 0.37),
('106', 'UCAC1')   : (0.40, 0.39),
('143', 'UCAC1')   : (0.57, 0.47),

# UCAC2
('703', 'UCAC2')   : (0.63, 0.59),
('G96', 'UCAC2')   : (0.32, 0.27),
('E12', 'UCAC2')   : (0.50, 0.45),
('683', 'UCAC2')   : (0.79, 0.90),
('J75', 'UCAC2')   : (0.41, 0.37),
('106', 'UCAC2')   : (0.40, 0.39),
('143', 'UCAC2')   : (0.57, 0.47),

# Gaia2
('T14', 'Gaia2')   : (0.10, 0.10),
('T12', 'Gaia2')   : (0.10, 0.10),
('T09', 'Gaia2')   : (0.10, 0.10),
('Y28', 'Gaia2')   : (0.30, 0.30),
('568', 'Gaia2')   : (0.10, 0.10),
('G83', 'Gaia2')   : (0.20, 0.20),
('309', 'Gaia2')   : (0.20, 0.20),

# Gaia3
('T14', 'Gaia3')   : (0.10, 0.10),
('T12', 'Gaia3')   : (0.10, 0.10),
('T09', 'Gaia3')   : (0.10, 0.10),
('Y28', 'Gaia3')   : (0.30, 0.30),
('568', 'Gaia3')   : (0.10, 0.10),
('G83', 'Gaia3')   : (0.20, 0.20),
('309', 'Gaia3')   : (0.20, 0.20),

# Gaia3E
('T14', 'Gaia3E')  : (0.10, 0.10),
('T12', 'Gaia3E')  : (0.10, 0.10),
('T09', 'Gaia3E')  : (0.10, 0.10),
('Y28', 'Gaia3E')  : (0.30, 0.30),
('568', 'Gaia3E')  : (0.10, 0.10),
('G83', 'Gaia3E')  : (0.20, 0.20),
('309', 'Gaia3E')  : (0.20, 0.20),

# Tycho-2
('689', 'Tycho-2') : (0.20, 0.21),

}

VFCCAstcat = {
# (astcat) : (ra, dec) arcsec 

'Tycho-2'          : (0.24, 0.25),
'UCAC2'            : (0.53, 0.49),
'UCAC1'            : (0.53, 0.49),
'UCAC4'            : (0.30, 0.30),
'USNOB1'           : (0.48, 0.42),
'USNOA1'           : (0.72, 0.69),
'USNOA2'           : (0.61, 0.58),
'Gaia2'            : (0.20, 0.20),
'Gaia3'            : (0.20, 0.20),
'Gaia3E'           : (0.20, 0.20),
'ATLAS2'           : (0.20, 0.20),

}

VFCCStn = {
# (stn) : (ra, dec) arcsec 

'645'              : (0.30, 0.30),
'673'              : (0.30, 0.30),
'689'              : (0.50, 0.50),
'950'              : (0.50, 0.50),
'H01'              : (0.30, 0.30),
'J04'              : (0.40, 0.40),
'W84'              : (0.50, 0.50),
'LCO'              : (0.40, 0.40),

}

In [26]:
VFCCLookup = {
    
    "Default" : VFCCLookupDefault,
    "Astcat"  : VFCCAstcat,
    "Station" : VFCCStn,
    
}

![Uncertainties - Veres et al.](https://cdn.hackclub.com/019d41f5-4f49-79fa-907e-4670c4fbb91b/Screenshot%202026-03-27%20173647.png)

![Uncertainties - Veres et al.](https://cdn.hackclub.com/019d41f5-e298-71e5-8a0f-a76e04ae62f4/Screenshot%202026-03-27%20215631.png)

![Uncertainties - ATLAS2 Paper](https://cdn.hackclub.com/019d41f6-4bb3-7d34-9d56-997bc03d7b66/Screenshot%202026-03-27%20222123.png)

In [27]:
def loadVFCC(details, dec_obs, lookup=VFCCLookup):
    
    if (details["stn"], details["astcat"]) in VFCCLookup["Default"]:
        
        sigma_ra  = VFCCLookup["Default"][(details["stn"], details["astcat"])][0]
        sigma_dec = VFCCLookup["Default"][(details["stn"], details["astcat"])][1]
        
    elif details["stn"] in VFCCLookup["Station"] and details["astcat"] in VFCCLookup["Astcat"]:
        
        sigma_ra  = VFCCLookup["Astcat"][details["astcat"]][0]
        sigma_dec = VFCCLookup["Astcat"][details["astcat"]][1]
        
        if sigma_ra < VFCCLookup["Station"][details["stn"]][0]:
            sigma_ra = VFCCLookup["Station"][details["stn"]][0]
            
        if sigma_dec < VFCCLookup["Station"][details["stn"]][1]:
            sigma_dec = VFCCLookup["Station"][details["stn"]][1]
        
    elif details["astcat"] in VFCCLookup["Astcat"]:
        
        sigma_ra  = VFCCLookup["Astcat"][details["astcat"]][0]
        sigma_dec = VFCCLookup["Astcat"][details["astcat"]][1]
        
    elif details["stn"] in VFCCLookup["Station"]:
        
        sigma_ra  = VFCCLookup["Station"][details["stn"]][0]
        sigma_dec = VFCCLookup["Station"][details["stn"]][1]
        
    else:
        
        sigma = 9.69627362219072e-07 # 0.2 * arcsec_to_rad # estimate
        
        sigma_ra  = sigma * np.cos(dec_obs)
        sigma_dec = sigma
        
    return {"sigma_ra": sigma_ra, "sigma_dec": sigma_dec}

In [28]:
def astrometric_error(details, dec_obs, arcsec_to_rad):
    
    sigma_ra  = None
    sigma_dec = None
    
    def rms(sigma_ra, sigma_dec):
        
        return {
            "rmsra"  : 1 / sigma_ra ** 2,
            "rmsdec" : 1 / sigma_dec ** 2}
            
    def largeMPCObserverError(details):
        
        sigma_ra  = float(details["rmsra"])
        sigma_dec = float(details["rmsdec"])
        
        if sigma_ra <= 0.2 and sigma_dec <= 0.2:
            return False
        else:
            return True
            
    if details["rmsra"] is not None and details["rmsdec"] is not None:
        
        sigma_ra  = float(details["rmsra"])
        sigma_dec = float(details["rmsdec"])
            
        if largeMPCObserverError(details):
            
            VFCCUncertainties = loadVFCC(details, dec_obs)
            
            sigma_ra  = VFCCUncertainties["sigma_ra"]
            sigma_dec = VFCCUncertainties["sigma_dec"]
            
    else:
        
        VFCCUncertainties = loadVFCC(details, dec_obs)
            
        sigma_ra  = VFCCUncertainties["sigma_ra"]
        sigma_dec = VFCCUncertainties["sigma_dec"]
        
    sigma_ra_rad  = sigma_ra * arcsec_to_rad
    sigma_dec_rad = sigma_dec * arcsec_to_rad
        
    rms_values = rms(sigma_ra=sigma_ra_rad, sigma_dec=sigma_dec_rad)
    
    return [rms_values["rmsra"], rms_values["rmsdec"]]

In [29]:
def angle_diff(a, b):
    return (a - b + np.pi) % (2*np.pi) - np.pi

In [30]:
def ObservationalError(obs, trajectory_solution=trajectory_solution):

    t_obs   = obs['obstime']
    ra_obs  = np.deg2rad(float(obs['ra']))
    dec_obs = np.deg2rad(float(obs['dec']))
    
    state_at_obs = trajectory_solution(t_obs)
    r_asteroid = state_at_obs[:3]
    
    stn    = obs['stn']
    astcat = obs['astcat']
    
    stn_properties = get_stn_properties(stn)
    obs_pos = 0.0
    
    if stn_properties['sin'] == None and stn_properties['cos'] == None:
        earth, _ = sp.spkezr('EARTH', t_obs, 'J2000', 'NONE', 'SUN')
        obs_pos = earth[:3]
    else:
        obs_pos = get_observer_pos_j2000(stn, t_obs, stn_properties)
        
    lt = 0.0
        
    for _ in range(3):
        
        rho = r_asteroid - obs_pos
        lt = np.linalg.norm(rho)
        t_emit = t_obs - (lt / c_km_s)
        r_asteroid = trajectory_solution(t_emit)[:3]
        
    rho = r_asteroid - obs_pos
    rho_hat = rho / np.linalg.norm(rho)
    
    ra_pred = np.arctan2(rho_hat[1], rho_hat[0])
    ra_pred = np.mod(ra_pred, 2*np.pi)
    dec_pred = np.arcsin(rho_hat[2])
    
    residual = [angle_diff(ra_obs, ra_pred), dec_obs - dec_pred]
    
    rmsra = obs['rmsra']
    rmsdec = obs['rmsdec']
    
    telescope_details = {"stn": stn, "astcat": astcat, "rmsra": rmsra, "rmsdec": rmsdec}
    
    weight = astrometric_error(telescope_details, dec_obs, arcsec_to_rad)

    return residual, weight

In [31]:
print(len(obs_data[obs_index:]))

1590


In [32]:
observations = [obs for obs in obs_data[obs_index+1:] if obs['stn'] != '270'] #omits garbage obs 1 and garbage observations from stn 270
result = [ObservationalError(obs) for obs in observations]

In [33]:
print(result)

[([np.float64(0.002031980511595144), np.float64(0.00139856075335687)], [np.float64(1063629257403.8047), np.float64(1063629257403.8047)]), ([np.float64(1.7984684319571898e-07), np.float64(-1.6108873635012344e-07)], [np.float64(1063629257403.8047), np.float64(1063629257403.8047)]), ([np.float64(-1.3905824047810711e-08), np.float64(1.5360935301522716e-07)], [np.float64(1063629257403.8047), np.float64(1063629257403.8047)]), ([np.float64(-2.726191139501566e-08), np.float64(-4.949280973942649e-08)], [np.float64(1063629257403.8047), np.float64(1063629257403.8047)]), ([np.float64(6.311030920080896e-07), np.float64(5.464328700632848e-08)], [np.float64(1063629257403.8047), np.float64(1063629257403.8047)]), ([np.float64(6.612491487700822e-07), np.float64(-8.907495174792501e-08)], [np.float64(1063629257403.8047), np.float64(1063629257403.8047)]), ([np.float64(4.901996466522007e-07), np.float64(-2.7454224105216696e-07)], [np.float64(1063629257403.8047), np.float64(1063629257403.8047)]), ([np.float6

In [34]:
residual_states, weights = zip(*result)
residual_states = np.array(residual_states, dtype=np.float64)
weights         = np.array(weights, dtype=np.float64)

![](https://cdn.hackclub.com/019d41f7-60ad-7f51-8a29-110340875089/Screenshot%202026-03-24%20160634.png)

In [35]:
e = residual_states.flatten().reshape(-1, 1) # 2*sum(obs_index:), 1
e_t = e.T # 1, 2*sum(obs_index:)

W = np.diag(weights.flatten()) # 2*sum(obs_index:), 2*sum(obs_index:)

q = e_t @ W @ e #1, 1

Q = float(q[0, 0])

print(Q)

6550136.6577333845


## Tests

In [36]:
print(np.mean(np.abs(residual_states)) / arcsec_to_rad)

0.47024272899068037
